# nugie — training a code LLM on Kaggle (2 × NVIDIA T4)

This notebook runs the **[nugie](https://github.com/wisnunugroho21/nugie-coding-llm-agent-4)**
project end to end on Kaggle using **two T4 GPUs**. It uses the repo's built-in
`t4x2` device preset, which replicates the model and shards each batch across both
GPUs with `nnx.pmap` (JAX data parallelism), all-reducing gradients and MoE expert
load every step.

**Pipeline:** RefineCode data → **pretrain** (WSD) → **anneal** (WSD decay) →
**SFT** (response-only) → greedy sample.

---
### ⚙️ Before you run
1. Right sidebar → **Session options → Accelerator → `GPU T4 x2`**.
2. Turn **Internet: On** (needed to clone the repo and install JAX CUDA wheels).
3. Run the cells top to bottom.


## 1. Confirm both T4 GPUs are visible

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 2. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/wisnunugroho21/nugie-coding-llm-agent-4.git"
REPO_DIR = "/kaggle/working/nugie-coding-llm-agent-4"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

## 3. Install dependencies (JAX with CUDA 12)

`jax[cuda12]` pulls the CUDA runtime as pip wheels, so it works on Kaggle without
touching the system CUDA. `flax` (nnx), `optax`, `tokenizers`, and `numpy` complete
the training stack.

In [ ]:
!pip install -q -U "jax[cuda12]" "flax>=0.10" "optax>=0.2" "tokenizers>=0.15" "numpy>=1.23"

## 4. Verify JAX sees **2** GPU devices

If this prints only one device, go back and set the accelerator to **GPU T4 x2**,
then *Factory reset* the session. The `t4x2` preset falls back to single-device
automatically, but you want both here.

In [ ]:
import jax
print("JAX version:", jax.__version__)
print("Devices:", jax.devices())
print("Local device count:", jax.local_device_count())
assert jax.local_device_count() >= 1, "No accelerator found"
if jax.local_device_count() < 2:
    print("\n[warning] Fewer than 2 GPUs visible — set Accelerator to 'GPU T4 x2'.")
else:
    print("\nOK: data parallelism will shard batches across", jax.local_device_count(), "GPUs.")

## 5. Build the sample data corpus

`make_sample_data.py` writes raw code files; the RefineCode pipeline then cleans,
dedups, filters, and downsamples them into `sample_data/refined.jsonl` — the
pretraining corpus.

In [ ]:
!python scripts/make_sample_data.py
!python -m data_pipeline.cli run --input sample_data/raw_code.jsonl --output sample_data/refined.jsonl

## 6. Build the annealing mixture + SFT data

Reuses the `annealing` and `sft` stages (offline, no API key) to produce the
`anneal` and `sft` phase inputs. (We build these inline rather than importing the
demo script, which pins JAX to CPU.)

In [ ]:
import os, json
from annealing import AnnealingConfig, build_annealing_data
from data_pipeline.io_utils import read_jsonl, write_jsonl
from sft import SFTConfig, synthesize_educational, synthesize_package
from sft.config import PackageSynthConfig

DATA = "sample_data"
docs = list(read_jsonl(os.path.join(DATA, "refined.jsonl")))

SNIPPET_SEEDS = [
    'def gcd(a, b):\n    """Greatest common divisor."""\n    while b:\n        a, b = b, a % b\n    return a\n',
    "def flatten(xss):\n    return [x for xs in xss for x in xs]\n",
]

# Annealing mixture -> content JSONL
mix, _ = build_annealing_data(docs, SNIPPET_SEEDS, total_token_budget=8000, cfg=AnnealingConfig())
write_jsonl(os.path.join(DATA, "annealing.jsonl"), mix)

# SFT instruction/response JSONL
cfg = SFTConfig()
examples = list(synthesize_educational(SNIPPET_SEEDS, cfg.educational))
examples += list(synthesize_package(
    PackageSynthConfig(libraries=("math", "json"), max_apis_per_library=6)))
with open(os.path.join(DATA, "sft_demo.jsonl"), "w", encoding="utf-8") as fh:
    for ex in examples:
        fh.write(json.dumps({"instruction": ex.instruction, "response": ex.response}) + "\n")

print(f"annealing docs: {len(mix)}  ->  {DATA}/annealing.jsonl")
print(f"SFT examples:   {len(examples)}  ->  {DATA}/sft_demo.jsonl")

## 7. Inspect the `t4x2` device preset

`d_model=512`, 8 layers, 8 MoE experts, bf16 compute, seq 1024. Per-device batch 4
× grad-accum 8 × **2 GPUs** ≈ global batch 64.

In [ ]:
!python -m training.devices

## 8. Train — pretrain → anneal → SFT (data-parallel on 2×T4)

Each phase warm-starts from the previous checkpoint. `--device t4x2` selects the
2-GPU data-parallel preset; the batch is split across both T4s and gradients are
all-reduced every step (look for the `data-parallel across 2 devices` log line).

`--steps` here is kept modest so the notebook finishes within Kaggle's session
limit. **Scale these up** (and point `--data` at a real corpus, with a BPE
`--tokenizer`) for a serious run.

In [ ]:
import os
os.makedirs("ckpt", exist_ok=True)

In [ ]:
# Phase 1 — pretrain (WSD warmup + stable) on the RefineCode corpus
!python -m training.cli pretrain --device t4x2 \
    --data sample_data/refined.jsonl \
    --steps 200 --log-every 20 --save ckpt/pretrain.pkl

In [ ]:
# Phase 2 — anneal (WSD decay -> 1e-5), warm-started from pretrain
!python -m training.cli anneal --device t4x2 \
    --data sample_data/annealing.jsonl \
    --init ckpt/pretrain.pkl --steps 100 --log-every 20 --save ckpt/anneal.pkl

In [ ]:
# Phase 3 — SFT stage 1 (cosine, response-only loss), warm-started from anneal
!python -m training.cli sft --stage 1 --device t4x2 \
    --data sample_data/sft_demo.jsonl \
    --init ckpt/anneal.pkl --steps 100 --log-every 20 --save ckpt/sft1.pkl

## 9. Generate a sample from the trained model

Loads the SFT checkpoint into the `t4x2` model config and greedily decodes. The
byte-level demo model is tiny and lightly trained, so the output is mostly
gibberish — the point is that the full multi-GPU pipeline runs end to end.

In [ ]:
import flax.nnx as nnx
import jax.numpy as jnp
import numpy as np

from training import KimiLinear, load_model
from training.devices import get_preset
from training.tokenizer import ByteTokenizer

mcfg = get_preset("t4x2").model
model = KimiLinear(mcfg, rngs=nnx.Rngs(0))
load_model(model, "ckpt/sft1.pkl")

tok = ByteTokenizer()
prompt = ("<|system|>\nYou are a helpful programming assistant.\n"
          "<|user|>\nWrite a Python function.\n<|assistant|>\n")
ids = tok.encode(prompt)
chunk = mcfg.gdn_chunk_size
if len(ids) % chunk:                       # pad to a GDN-2 chunk multiple for prefill
    ids = ids + tok.encode(" ") * (chunk - len(ids) % chunk)

out = model.generate(jnp.asarray([ids], jnp.int32), max_new_tokens=64)
print("PROMPT:", prompt.replace("\n", "\\n"))
print("SAMPLE:", repr(tok.decode(list(np.asarray(out[0])))))

## 10. (Optional) Scale up

To move from demo to a real run, keep `--device t4x2` and change the inputs:

```bash
# Train a real byte-level BPE vocab from the corpus, then train with it
python scripts/train_tokenizer.py --data sample_data/refined.jsonl \
    --vocab-size 32000 --output tok.json
python -m training.cli pretrain --device t4x2 --data <big_corpus>.jsonl \
    --tokenizer tok.json --steps 20000 --save ckpt/pretrain.pkl
```

- Raise `--steps` per phase (pretrain dominates).
- Point `--data` at the real RefineCode corpus (see `data_ingestion/` to ingest
  The Stack v2), and pass the same `--tokenizer` to every phase.
- Kaggle sessions are time-limited — save checkpoints to `/kaggle/working` (done
  here) so they persist as notebook output, and resume with `--init`.
